# Dataset Audit: VisDrone2019-DET

**Course:** Computer Vision Engineering for DefenceTech  
**Assignment:** Data & Baseline Report, Part 1 — Dataset Readiness  
**Author:** Serhii Polishchuk  

This notebook performs a full engineering audit of the VisDrone2019-DET dataset:
1. Download & basic metadata
2. Structure and annotation validation
3. Visualization of annotations
4. Statistical distributions
5. Leakage risk checks
6. Domain shift notes

The results of this notebook are used in the accompanying report.

## 0. Setup

In [ ]:
import os
import zipfile
import urllib.request
from pathlib import Path
from collections import Counter, defaultdict
import json
import random

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm

# Reproducibility
random.seed(42)
np.random.seed(42)

# Paths
DATA_ROOT = Path("./datasets/VisDrone")
DATA_ROOT.mkdir(parents=True, exist_ok=True)

print("Setup complete.")
print(f"Data root: {DATA_ROOT.resolve()}")

## 1. Download VisDrone2019-DET

We use the official Ultralytics-hosted mirrors (same data as the original VisDrone challenge).

In [ ]:
URLS = {
    "VisDrone2019-DET-train": "https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-train.zip",
    "VisDrone2019-DET-val":   "https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-val.zip",
    "VisDrone2019-DET-test-dev": "https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-test-dev.zip",
}

def download_and_extract(name: str, url: str, root: Path):
    target_dir = root / name
    zip_path = root / f"{name}.zip"

    if target_dir.exists() and any(target_dir.rglob("*.jpg")):
        print(f"[SKIP] {name} already exists")
        return

    print(f"[DOWNLOAD] {name} ...")
    urllib.request.urlretrieve(url, zip_path)
    print(f"[EXTRACT]  {name} ...")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(root)
    zip_path.unlink()  # remove zip to save space
    print(f"[DONE]    {name}")

for name, url in URLS.items():
    download_and_extract(name, url, DATA_ROOT)

print("\nDownload finished.")
print("Contents of data root:")
for p in sorted(DATA_ROOT.iterdir()):
    print(" -", p.name)

## 2. Dataset Metadata (Section 1 of the Report)

Fix source, version, splits, counts, file structure and license.

In [ ]:
# Official class mapping (VisDrone2019-DET)
CLASS_NAMES = {
    0: "ignored regions",
    1: "pedestrian",
    2: "people",
    3: "bicycle",
    4: "car",
    5: "van",
    6: "truck",
    7: "tricycle",
    8: "awning-tricycle",
    9: "bus",
    10: "motor",
    11: "others",
}

# YOLO-style mapping (used by Ultralytics conversion, classes 0-9 after dropping ignored)
YOLO_CLASS_NAMES = {
    0: "pedestrian",
    1: "people",
    2: "bicycle",
    3: "car",
    4: "van",
    5: "truck",
    6: "tricycle",
    7: "awning-tricycle",
    8: "bus",
    9: "motor",
}

SPLITS = {
    "train": DATA_ROOT / "VisDrone2019-DET-train",
    "val":   DATA_ROOT / "VisDrone2019-DET-val",
    "test-dev": DATA_ROOT / "VisDrone2019-DET-test-dev",
}

def count_images_and_annotations(split_dir: Path):
    images_dir = split_dir / "images"
    ann_dir = split_dir / "annotations"

    n_images = len(list(images_dir.glob("*.jpg"))) if images_dir.exists() else 0
    n_ann_files = len(list(ann_dir.glob("*.txt"))) if ann_dir.exists() else 0

    n_boxes = 0
    if ann_dir.exists():
        for f in ann_dir.glob("*.txt"):
            with open(f) as fh:
                n_boxes += sum(1 for _ in fh if _.strip())

    return {
        "images": n_images,
        "annotation_files": n_ann_files,
        "bounding_boxes": n_boxes,
    }

meta = {}
for split, path in SPLITS.items():
    meta[split] = count_images_and_annotations(path)

meta_df = pd.DataFrame(meta).T
print("=== Dataset counts ===")
display(meta_df)

print("\n=== Official information ===")
print("Source:      VisDrone2019-DET (AISKYEYE team, Tianjin University)")
print("Version:     VisDrone2019 (object detection in images task)")
print("Paper:       Zhu et al., IEEE TPAMI 2021")
print("License:     Research / academic use (see official VisDrone site)")
print("Splits used: train / val / test-dev (official)")
print("Classes:     10 main object classes + ignored regions + others")

In [ ]:
# File structure overview
print("=== Directory structure (example for train) ===")
train_path = SPLITS["train"]
for item in sorted(train_path.iterdir()):
    if item.is_dir():
        n = len(list(item.glob("*")))
        print(f"  {item.name}/  ({n} files)")
    else:
        print(f"  {item.name}")

## 3. Automatic Structure & Annotation Checks

We validate:
- every image has a corresponding annotation file
- annotation format (bbox coordinates, class id, truncation, occlusion)
- boxes are inside image boundaries
- class ids are in the expected range

In [ ]:
def parse_annotation_line(line: str):
    """Parse one VisDrone annotation line.
    Format: <bbox_left>,<bbox_top>,<bbox_width>,<bbox_height>,<score>,<object_category>,<truncation>,<occlusion>
    """
    parts = line.strip().split(",")
    if len(parts) < 8:
        return None
    try:
        return {
            "x": int(parts[0]),
            "y": int(parts[1]),
            "w": int(parts[2]),
            "h": int(parts[3]),
            "score": int(parts[4]),
            "category": int(parts[5]),
            "truncation": int(parts[6]),
            "occlusion": int(parts[7]),
        }
    except ValueError:
        return None


def audit_split(split_name: str, split_dir: Path, max_problems_to_store=200):
    images_dir = split_dir / "images"
    ann_dir = split_dir / "annotations"

    problems = []
    stats = {
        "images": 0,
        "ann_files": 0,
        "boxes_total": 0,
        "ignored_regions": 0,
        "valid_objects": 0,
        "class_counts": Counter(),
        "occlusion_counts": Counter(),
        "truncation_counts": Counter(),
        "box_areas": [],
        "boxes_per_image": [],
    }

    image_files = {p.stem: p for p in images_dir.glob("*.jpg")}
    ann_files = {p.stem: p for p in ann_dir.glob("*.txt")}

    stats["images"] = len(image_files)
    stats["ann_files"] = len(ann_files)

    # Missing pairs
    missing_ann = set(image_files) - set(ann_files)
    missing_img = set(ann_files) - set(image_files)
    for stem in list(missing_ann)[:50]:
        problems.append({"split": split_name, "type": "missing_annotation", "file": stem, "detail": "image without .txt"})
    for stem in list(missing_img)[:50]:
        problems.append({"split": split_name, "type": "missing_image", "file": stem, "detail": "annotation without .jpg"})

    common = set(image_files) & set(ann_files)

    for stem in tqdm(list(common), desc=f"Auditing {split_name}"):
        img_path = image_files[stem]
        ann_path = ann_files[stem]

        # Image size
        try:
            with Image.open(img_path) as im:
                W, H = im.size
        except Exception as e:
            problems.append({"split": split_name, "type": "corrupt_image", "file": stem, "detail": str(e)})
            continue

        boxes_in_img = 0
        with open(ann_path) as fh:
            for line_no, line in enumerate(fh, 1):
                if not line.strip():
                    continue
                box = parse_annotation_line(line)
                if box is None:
                    problems.append({"split": split_name, "type": "malformed_line", "file": stem, "detail": f"line {line_no}"})
                    continue

                stats["boxes_total"] += 1
                boxes_in_img += 1

                cat = box["category"]
                stats["class_counts"][cat] += 1
                stats["occlusion_counts"][box["occlusion"]] += 1
                stats["truncation_counts"][box["truncation"]] += 1

                if cat == 0:
                    stats["ignored_regions"] += 1
                else:
                    stats["valid_objects"] += 1
                    stats["box_areas"].append(box["w"] * box["h"])

                # Geometry checks
                if box["w"] <= 0 or box["h"] <= 0:
                    problems.append({"split": split_name, "type": "zero_or_negative_size", "file": stem, "detail": f"w={box['w']}, h={box['h']}"})
                if box["x"] < 0 or box["y"] < 0:
                    problems.append({"split": split_name, "type": "negative_coords", "file": stem, "detail": f"x={box['x']}, y={box['y']}"})
                if box["x"] + box["w"] > W + 1 or box["y"] + box["h"] > H + 1:  # small tolerance
                    problems.append({"split": split_name, "type": "box_outside_image", "file": stem,
                                    "detail": f"box=({box['x']},{box['y']},{box['w']},{box['h']}), img=({W},{H})"})
                if cat not in CLASS_NAMES:
                    problems.append({"split": split_name, "type": "unknown_class", "file": stem, "detail": f"category={cat}"})

                if len(problems) >= max_problems_to_store:
                    break

        stats["boxes_per_image"].append(boxes_in_img)

        if len(problems) >= max_problems_to_store:
            problems.append({"split": split_name, "type": "truncated", "file": "-", "detail": "stopped after max_problems"})
            break

    return stats, problems


all_stats = {}
all_problems = []

for split, path in SPLITS.items():
    if not path.exists():
        print(f"[WARN] {split} not found, skipping")
        continue
    stats, problems = audit_split(split, path)
    all_stats[split] = stats
    all_problems.extend(problems)

problems_df = pd.DataFrame(all_problems)
print(f"\nTotal problems recorded: {len(problems_df)}")
if len(problems_df):
    display(problems_df["type"].value_counts())
    display(problems_df.head(20))

In [ ]:
# Summary table of problems (for the report)
if len(problems_df):
    summary = problems_df.groupby(["split", "type"]).size().reset_index(name="count")
    print("=== Problems summary ===")
    display(summary)
else:
    print("No structural problems found (or only truncated list).")

## 4. Visualization of Annotations

We look at:
- dense scenes
- small objects
- rare classes
- anomalous annotations

In [ ]:
COLORS = {
    1: (0, 255, 0),      # pedestrian - green
    2: (0, 200, 100),    # people
    3: (255, 165, 0),    # bicycle - orange
    4: (0, 0, 255),      # car - red
    5: (255, 0, 255),    # van - magenta
    6: (255, 255, 0),    # truck - cyan
    7: (128, 0, 128),    # tricycle
    8: (0, 128, 128),    # awning-tricycle
    9: (0, 128, 255),    # bus
    10: (128, 128, 0),   # motor
    0: (128, 128, 128),  # ignored
    11: (200, 200, 200), # others
}

def draw_annotations(img_path: Path, ann_path: Path, max_boxes=300):
    img = cv2.imread(str(img_path))
    if img is None:
        return None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    with open(ann_path) as fh:
        lines = [l for l in fh if l.strip()]

    for line in lines[:max_boxes]:
        box = parse_annotation_line(line)
        if box is None:
            continue
        x, y, w, h = box["x"], box["y"], box["w"], box["h"]
        cat = box["category"]
        color = COLORS.get(cat, (255, 255, 255))
        cv2.rectangle(img, (x, y), (x + w, y + h), color, 2)
        label = CLASS_NAMES.get(cat, str(cat))
        cv2.putText(img, label, (x, max(y - 5, 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)

    return img


def show_examples(split_name="train", n=4, selection="random"):
    split_dir = SPLITS[split_name]
    images_dir = split_dir / "images"
    ann_dir = split_dir / "annotations"

    stems = list({p.stem for p in images_dir.glob("*.jpg")} & {p.stem for p in ann_dir.glob("*.txt")})

    if selection == "dense":
        # pick images with many boxes
        counts = []
        for s in stems:
            with open(ann_dir / f"{s}.txt") as fh:
                n_boxes = sum(1 for _ in fh if _.strip())
            counts.append((n_boxes, s))
        counts.sort(reverse=True)
        chosen = [s for _, s in counts[:n]]
    else:
        chosen = random.sample(stems, min(n, len(stems)))

    fig, axes = plt.subplots(1, len(chosen), figsize=(5 * len(chosen), 5))
    if len(chosen) == 1:
        axes = [axes]

    for ax, stem in zip(axes, chosen):
        img = draw_annotations(images_dir / f"{stem}.jpg", ann_dir / f"{stem}.txt")
        if img is not None:
            ax.imshow(img)
        ax.set_title(stem, fontsize=9)
        ax.axis("off")

    plt.suptitle(f"{split_name} — {selection} examples", fontsize=14)
    plt.tight_layout()
    plt.show()


print("Random examples from train:")
show_examples("train", n=3, selection="random")

print("Dense scenes from train:")
show_examples("train", n=3, selection="dense")

## 5. Distributions (Section 2 of the Report)

In [ ]:
# Class distribution (train)
train_stats = all_stats.get("train")
if train_stats:
    class_counts = train_stats["class_counts"]
    # filter ignored for the main plot
    main_classes = {k: v for k, v in class_counts.items() if k not in (0, 11)}

    labels = [CLASS_NAMES.get(k, str(k)) for k in sorted(main_classes)]
    values = [main_classes[k] for k in sorted(main_classes)]

    fig, ax = plt.subplots(figsize=(10, 5))
    sns.barplot(x=labels, y=values, ax=ax, hue=labels, legend=False, palette="viridis")
    ax.set_title("Class distribution (train, excluding ignored/others)")
    ax.set_ylabel("Number of boxes")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

    print("Raw class counts (train):")
    for k in sorted(class_counts):
        print(f"  {k:2d} {CLASS_NAMES.get(k, '?'):20s}: {class_counts[k]:,}")

In [ ]:
# Object size distribution
if train_stats and train_stats["box_areas"]:
    areas = np.array(train_stats["box_areas"])
    sqrt_areas = np.sqrt(areas)  # approx side length

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].hist(areas, bins=80, log=True, color="steelblue", edgecolor="white")
    axes[0].set_title("Box area distribution (log scale)")
    axes[0].set_xlabel("Area (pixels²)")
    axes[0].set_ylabel("Count")

    axes[1].hist(sqrt_areas, bins=80, color="coral", edgecolor="white")
    axes[1].set_title("Approx. object side length (sqrt(area))")
    axes[1].set_xlabel("Side length (pixels)")

    plt.tight_layout()
    plt.show()

    print(f"Median area: {np.median(areas):.0f} px²")
    print(f"Median side: {np.median(sqrt_areas):.1f} px")
    print(f"% of boxes with side < 20 px: {(sqrt_areas < 20).mean()*100:.1f}%")
    print(f"% of boxes with side < 32 px: {(sqrt_areas < 32).mean()*100:.1f}%")

In [ ]:
# Objects per image
if train_stats and train_stats["boxes_per_image"]:
    bpi = np.array(train_stats["boxes_per_image"])

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(bpi, bins=50, color="seagreen", edgecolor="white")
    ax.set_title("Number of objects per image (train)")
    ax.set_xlabel("Objects per image")
    ax.set_ylabel("Number of images")
    plt.tight_layout()
    plt.show()

    print(f"Mean objects/image: {bpi.mean():.1f}")
    print(f"Median: {np.median(bpi):.0f}")
    print(f"Max: {bpi.max()}")
    print(f"Images with > 100 objects: {(bpi > 100).sum()}")

In [ ]:
# Occlusion & truncation (if available)
if train_stats:
    print("Occlusion levels (0=none, 1=partial, 2=heavy):")
    for k, v in sorted(train_stats["occlusion_counts"].items()):
        print(f"  {k}: {v:,}")

    print("\nTruncation levels (0=none, 1=partial):")
    for k, v in sorted(train_stats["truncation_counts"].items()):
        print(f"  {k}: {v:,}")

## 6. Leakage Risk Checks (Section 4 of the Report)

We keep the official train/val/test-dev splits.

Checks performed:
1. Exact duplicate images (by file hash / size+name)
2. Near-duplicate search (simple perceptual approach on a sample)

**Limitations** (to be stated in the report):
- Full pairwise near-duplicate search on 6k+ images is expensive; we use sampling.
- We do not check video-frame continuity across different splits (VisDrone is mostly static images for DET).

In [ ]:
import hashlib

def file_md5(path: Path, chunk=1<<20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        while True:
            data = f.read(chunk)
            if not data:
                break
            h.update(data)
    return h.hexdigest()


def find_exact_duplicates(split_dirs: dict, sample_limit=None):
    """Return groups of identical files by MD5."""
    hash_to_files = defaultdict(list)

    for split, path in split_dirs.items():
        images = list((path / "images").glob("*.jpg"))
        if sample_limit:
            images = images[:sample_limit]
        for img in tqdm(images, desc=f"Hashing {split}"):
            h = file_md5(img)
            hash_to_files[h].append((split, img.name))

    duplicates = {h: files for h, files in hash_to_files.items() if len(files) > 1}
    return duplicates


# Exact duplicates across train + val (full scan is feasible)
print("Searching for exact duplicate images between train and val...")
exact_dups = find_exact_duplicates({"train": SPLITS["train"], "val": SPLITS["val"]})

print(f"Found {len(exact_dups)} groups of exact duplicates.")
if exact_dups:
    for i, (h, files) in enumerate(list(exact_dups.items())[:10]):
        print(f"  Group {i+1}: {files}")
else:
    print("No exact duplicates found between train and val.")

In [ ]:
# Simple near-duplicate check on a sample (average hash)
def average_hash(img_path: Path, hash_size=8):
    img = Image.open(img_path).convert("L").resize((hash_size, hash_size), Image.Resampling.LANCZOS)
    pixels = list(img.getdata())
    avg = sum(pixels) / len(pixels)
    bits = "".join("1" if p > avg else "0" for p in pixels)
    return bits


def hamming(a, b):
    return sum(c1 != c2 for c1, c2 in zip(a, b))


print("Near-duplicate search (sample of 300 train + all val)...")

train_imgs = list((SPLITS["train"] / "images").glob("*.jpg"))
val_imgs = list((SPLITS["val"] / "images").glob("*.jpg"))

train_sample = random.sample(train_imgs, min(300, len(train_imgs)))

train_hashes = {p.name: average_hash(p) for p in tqdm(train_sample, desc="train hashes")}
val_hashes = {p.name: average_hash(p) for p in tqdm(val_imgs, desc="val hashes")}

near_pairs = []
for t_name, t_hash in train_hashes.items():
    for v_name, v_hash in val_hashes.items():
        dist = hamming(t_hash, v_hash)
        if dist <= 5:  # very similar
            near_pairs.append((t_name, v_name, dist))

print(f"Near-duplicate pairs (Hamming ≤ 5): {len(near_pairs)}")
for pair in near_pairs[:15]:
    print(" ", pair)

## 7. Domain Shift Notes (Section 5 of the Report)

VisDrone2019-DET was collected by drones in Chinese cities under various weather and lighting conditions.

**Typical future application scenarios** (DefenceTech / security):
- different geographic region (Europe / Middle East / Africa)
- different altitude / camera resolution / FOV
- military vehicles / different vehicle types not present in VisDrone
- night / thermal / low-light conditions
- heavy occlusion by trees / buildings / camouflage

Unknown parameters that should be fixed later:
- exact camera model and altitude distribution of the target system
- expected object size distribution in the target domain
- presence of classes that do not exist in VisDrone (e.g. tanks, UAVs themselves)

## 8. Quick Summary for the Report

Run this cell after all previous cells to get a compact overview.

In [ ]:
print("=" * 60)
print("DATASET AUDIT SUMMARY — VisDrone2019-DET")
print("=" * 60)

print("\n1. Counts:")
display(meta_df)

print("\n2. Structural problems found:", len(problems_df))
if len(problems_df):
    print(problems_df["type"].value_counts().to_string())

print("\n3. Exact duplicates (train↔val):", len(exact_dups))
print("4. Near-duplicate pairs (sample):", len(near_pairs))

if train_stats:
    areas = np.array(train_stats["box_areas"])
    print(f"\n5. Small objects (side < 32 px): {(np.sqrt(areas) < 32).mean()*100:.1f}%")
    print(f"6. Mean objects per image: {np.mean(train_stats['boxes_per_image']):.1f}")

print("\nReady for report writing.")